# Entry Strategy Analysis

Read the latest notifications from a Telegram channel. Telegram configuration is read directly from the process environment.

In [2]:
import os
import re

from IPython.display import display
from telethon import TelegramClient, types, utils
from telethon.sessions import StringSession
import pandas as pd 

def required_environment_variable(name):
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError(f"{name} must be set in the process environment")
    return value

def normalize_channel_reference(channel):
    channel = channel.strip()
    if not channel:
        raise ValueError("TELEGRAM_CHANNEL cannot be empty")
    try:
        channel_id = int(channel)
    except ValueError:
        return channel
    if channel_id == 0:
        raise ValueError("Numeric Telegram channel ID cannot be zero")
    if channel_id <= -1_000_000_000_000:
        return channel_id
    return utils.get_peer_id(types.PeerChannel(abs(channel_id)))

## Inputs

`CHANNEL` defaults to `TELEGRAM_CHANNEL`, but may be replaced with a username, `t.me` URL, or numeric channel ID.

In [3]:
TELEGRAM_API_ID = int(required_environment_variable("TELEGRAM_API_ID"))
TELEGRAM_API_HASH = required_environment_variable("TELEGRAM_API_HASH")
TELEGRAM_SESSION = required_environment_variable("TELEGRAM_SESSION")
TELEGRAM_PHONE = os.getenv("TELEGRAM_PHONE", "").strip() or None

CHANNEL = required_environment_variable("TELEGRAM_CHANNEL")
NOTIFICATION_COUNT = 300

if NOTIFICATION_COUNT <= 0:
    raise ValueError("NOTIFICATION_COUNT must be positive")

channel = normalize_channel_reference(CHANNEL)

In [4]:
async def read_latest_notifications(channel, limit):
    client = TelegramClient(
        StringSession(TELEGRAM_SESSION),
        TELEGRAM_API_ID,
        TELEGRAM_API_HASH,
    )

    try:
        await client.start(phone=TELEGRAM_PHONE)
        messages = await client.get_messages(channel, limit=limit)
        return [
            {
                "message_id": message.id,
                "date": message.date,
                "text": message.raw_text or "",
                "sender_id": message.sender_id,
                "has_media": message.media is not None,
                "grouped_id": message.grouped_id,
            }
            for message in messages
        ]
    finally:
        await client.disconnect()

In [5]:
notifications = await read_latest_notifications(channel, NOTIFICATION_COUNT)

## KRW market listings

Keep trading notices that either announce new trading support including the `KRW` market or explicitly add a digital asset to the `KRW` market, then extract each asset name and ticker from the headline.

In [6]:
def extract_krw_listing_assets(text):
    headline = text.splitlines()[0].strip() if text else ""

    # 거래 = "Trading"
    if headline.startswith("[거래]"): 
        # 신규 거래지원 안내 = "New trading support announcement"
        asset_section, separator, market_section = headline.partition("신규 거래지원 안내")
        if separator:
            market_match = re.search(r"\(([^()]*)\)\s*$", market_section)
            if not market_match:
                return []
            
            markets = set(re.findall(r"\b[A-Z]{2,10}\b", market_match.group(1)))
            if "KRW" not in markets:
                return []
            
        else:
            # KRW 마켓 디지털 자산 추가 = "Digital asset added to the KRW market"
            addition_phrase = re.search(r"KRW\s*마켓\s*디지털\s*자산\s*추가", headline)
            if not addition_phrase:
                return []
            asset_section = headline[:addition_phrase.start()]

        # 거래 = "Trading"
        asset_section = asset_section.removeprefix("[거래]").strip()

    else:
        return []

    asset_list = []
    for match in re.finditer(
        r"(?:^|,\s*)(?P<asset_name>[^,()]+?)\s*\((?P<symbol>[A-Z0-9]+)\)",
        asset_section,
    ):
        asset_list.append(match.group("symbol"))
    
    return asset_list

In [10]:
asset_list = []

for notification in notifications:
    assets = extract_krw_listing_assets(notification["text"])
    if not assets:
        continue

    for asset in assets: 
        asset_list.append({"asset": asset, "notification_time":notification['date'].isoformat()})

df = pd.DataFrame(asset_list)
df.to_csv("data/notification_history.csv", index=False)